In [ ]:
""" ОБРАБОТКА ЭКСПЕРИМЕНТАЛЬНЫХ ДАННЫХ


Функция для расчёта коэффициентов наклона и сдвига для экспериментальных данных


"""
import cv2
import numpy as np
import modulesDataSet as mds



lam = 532 * 10 ** (-9)
z = 0.200 #0.07
pixelsizeSLM = 8 * 10 ** (-6)

amplitude = np.ones((1024, 1024))
phase = np.zeros((1024, 1024))


def difraction(filepath):
  
  doe = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)

  doe = doe/255*2*np.pi

  E0 = amplitude * np.exp(1j * (phase + doe)) #комплексное поле до регистрации

  difracted = mds.detectCamera(mds.FresnelPropagator(E0, pixelsizeSLM, lam, z))
  difracted = np.sqrt(difracted)
  difracted = difracted/np.max(difracted) * 255

  return difracted

difracted = difraction('C:/Users/thatm/VSCodeProjects/experiment/trans.png')

image = cv2.imread('C:/Users/thatm/VSCodeProjects/experiment/data/0.png', cv2.IMREAD_GRAYSCALE)
image = cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
image = cv2.flip(image, 1)  # 0 - вертикальное отражение
#image = image[1300:3300, 400:2400]



def find_cross_dimensions(image):
    # Преобразование изображения в 8-битное при необходимости
    if image.dtype != np.uint8:
        image_8bit = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    else:
        image_8bit = image
    
    # Бинаризация
    _, binary = cv2.threshold(image_8bit, 50, 255, cv2.THRESH_BINARY)
    
    # Поиск контуров
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None, None, None
    
    # Выбор самого большого контура
    contour = max(contours, key=cv2.contourArea)
    
    # Минимальный ограничивающий прямоугольник (повернутый)
    rect = cv2.minAreaRect(contour)
    box = cv2.boxPoints(rect)
    box = np.int32(box)
    
    # Угол наклона
    angle = rect[2]  # угол в градусах
    
    # Корректировка угла для диапазона [0, 90)
    if angle < -45:
        angle = 90 + angle
    
    # Размеры ограничивающего прямоугольника
    width = rect[1][0]
    height = rect[1][1]
    
    # Для определения ширины и высоты креста используем моменты изображения
    mask = np.zeros_like(image_8bit)
    cv2.drawContours(mask, [contour], -1, 255, -1)
    
    # Находим центральные линии с использованием проекций
    y_coords, x_coords = np.where(mask > 0)
    
    if len(y_coords) == 0 or len(x_coords) == 0:
        return None, None, None
    
    # Определяем горизонтальную и вертикальную проекции
    horizontal_projection = np.sum(mask, axis=0)
    vertical_projection = np.sum(mask, axis=1)
    
    # Находим не нулевые участки в проекциях
    hor_indices = np.where(horizontal_projection > 0)[0]
    vert_indices = np.where(vertical_projection > 0)[0]
    
    if len(hor_indices) == 0 or len(vert_indices) == 0:
        return None, None, None
    
    # Ширина и высота креста
    cross_height = vert_indices[-1] - vert_indices[0]
    cross_width = hor_indices[-1] - hor_indices[0]
    center_x = (hor_indices[0] + hor_indices[-1]) // 2
    center_y = (vert_indices[0] + vert_indices[-1]) // 2

    
    # Для более точного определения угла используем линейную регрессию по границам
    # Находим края креста
    edges = cv2.Canny(mask, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50, minLineLength=30, maxLineGap=10)
    
    angles = []
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if abs(x2 - x1) > 0:  # избегаем деления на ноль
                line_angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
                # Оставляем только углы близкие к 0 или 90 градусам (для креста)
                if abs(line_angle) < 20 or abs(abs(line_angle) - 90) < 20:
                    angles.append(line_angle)
    
    
    # Визуализация для отладки
    result = cv2.cvtColor(image_8bit, cv2.COLOR_GRAY2BGR) if len(image_8bit.shape) == 2 else image_8bit.copy()
    
    # Рисуем повернутый прямоугольник
    cv2.drawContours(result, [box], 0, (0, 255, 0), 2)
    
    # Рисуем центральные линии
    cv2.line(result, (hor_indices[0], np.mean(vert_indices).astype(int)),
             (hor_indices[-1], np.mean(vert_indices).astype(int)), (255, 0, 0), 2)
    cv2.line(result, (np.mean(hor_indices).astype(int), vert_indices[0]),
             (np.mean(hor_indices).astype(int), vert_indices[-1]), (0, 0, 255), 2)
    
    # Выводим информацию
    cv2.putText(result, f'Angle: {angle:.2f} deg', (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 6)
    cv2.putText(result, f'Width: {cross_width} px', (10, 60), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 6)
    cv2.putText(result, f'Height: {cross_height} px', (10, 90), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 6)
    
    return cross_width, cross_height, angle, result, center_x, center_y


# Использование функции
size_x_ok, size_y_ok, slope_coef_ok, result_img_ok, center_x_ok, center_y_ok = find_cross_dimensions(difracted)
size_x, size_y, slope_coef, result_img, center_x, center_y = find_cross_dimensions(image)
center_x = int(center_x)
center_y = int(center_y)

scaleK = size_x_ok/size_x
rotateK_degree = slope_coef_ok - slope_coef
print(rotateK_degree)


rotated_image = cv2.warpAffine(image, 
                              cv2.getRotationMatrix2D((center_x, center_y), -rotateK_degree, 1.0), # Первое! 
                              (image.shape[1], image.shape[0]))
scaled_image = cv2.resize(rotated_image, None, fx=scaleK, fy=scaleK, interpolation=cv2.INTER_LINEAR) # Второе!
size_x_scaled, size_y_scaled, slope_coef_scaled, result_img_scaled, center_x_scaled, center_y_scaled = find_cross_dimensions(scaled_image) 
finallied = scaled_image[center_y_scaled-512:center_y_scaled+512, center_x_scaled-512:center_x_scaled+512] # Третье!

def transform(image, center_x, center_y, rotateK_degree, center_x_scaled, center_y_scaled): #обработка всех остальных изображений
    rotated_image = cv2.warpAffine(image, 
                              cv2.getRotationMatrix2D((center_x, center_y), -rotateK_degree, 1.0), # Первое! 
                              (image.shape[1], image.shape[0]))
    scaled_image = cv2.resize(rotated_image, None, fx=scaleK, fy=scaleK, interpolation=cv2.INTER_LINEAR) # Второе! 
    finallied = scaled_image[center_y_scaled-512:center_y_scaled+512, center_x_scaled-512:center_x_scaled+512] # Третье!
    return finallied

image_exp = cv2.imread('C:/Users/thatm/VSCodeProjects/experiment/data/Aber_3.png', cv2.IMREAD_GRAYSCALE)
image_exp = cv2.rotate(image_exp, cv2.ROTATE_90_COUNTERCLOCKWISE)
image_exp = cv2.flip(image_exp, 1)  # 0 - вертикальное отражение
image_exp = transform(image_exp, center_x, center_y, rotateK_degree, center_x_scaled, center_y_scaled)

def save_normalized_1024_to_256(image, filename):
    # Нормализуем к 0-255 если нужно
    if image.dtype != np.uint8:
        img_8bit = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    else:
        img_8bit = image
    
    # Конвертируем в grayscale если цветное
    if len(img_8bit.shape) == 3:
        gray = cv2.cvtColor(img_8bit, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_8bit
    
    # Сохраняем уменьшенное изображение
    cv2.imwrite(filename, cv2.resize(gray, (256, 256)))

#image_exp = save_normalized_1024_to_256(image_exp, 'C:/Users/thatm/VSCodeProjects/experiment/data/Aber_3_sized.png')

#plt.imshow(image_exp)
#plt.show()

#plt.imshow(difracted - finallied)
#plt.show()



In [ ]:
def combine(filepath_half):
  #Расчёт трехканального изображения дифракции в эксперименте
  img_array = np.zeros((256, 256, 3), dtype=np.uint8)
  
  for k in range(3):
    image = cv2.imread(filepath_half + str(k+1) + '_06_sized.png', cv2.IMREAD_GRAYSCALE)  
    image = image/np.max(image) * 255

    img_array[..., k] = image  # k-й канал

  return img_array

filepath_half = 'C:/Users/thatm/VSCodeProjects/experiment/data/'
combined = combine(filepath_half)

Image.fromarray(combined).save(f'C:/Users/thatm/VSCodeProjects/experiment/data/norm06.png')